# GA for tuning the h-p

In this notebook, I will tune only the ize and the number of the layers.

## A collection of EA operators 

Here 
- CX changes the sizes 
- Mut changes the sizes 

- CX maintains the same number of layers 
- Mut changes (sometimes) the number of layers 

In [3]:
# Crossover functions

import random

# Example:
# aaa|aa 
# bbb|bb
# --> aaa|bb, bbb|aa
def single_point_crossover(parent1, parent2):
    if len(parent1) == 1 or len(parent2) == 1:
        return parent1, parent2  # No crossover if a parent has only one layer
    point = random.randint(1, min(len(parent1), len(parent2)) - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

# Example:
# aaa|aa|a
# bbb|bb|b
# --> aaa|bb|a, bbb|aa|b
def two_point_crossover(parent1, parent2):
    if len(parent1) < 2 or len(parent2) < 2:
        return parent1, parent2  # No crossover if either parent has less than 2 layers
    point1 = random.randint(1, min(len(parent1), len(parent2)) - 2)
    point2 = random.randint(point1 + 1, min(len(parent1), len(parent2)) - 1)
    child1 = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
    child2 = parent2[:point1] + parent1[point1:point2] + parent2[point2:]
    return child1, child2

# Random crossover point for each layer
def uniform_crossover(parent1, parent2):
    child1, child2 = [], []
    for p1, p2 in zip(parent1, parent2):
        if random.random() > 0.5:
            child1.append(p1)
            child2.append(p2)
        else:
            child1.append(p2)
            child2.append(p1)
    return child1, child2

# We are working with class, so it is not okay  
def arithmetic_crossover(parent1, parent2):
    alpha = random.random()
    child1 = [alpha * p1 + (1 - alpha) * p2 for p1, p2 in zip(parent1, parent2)]
    child2 = [(1 - alpha) * p1 + alpha * p2 for p1, p2 in zip(parent1, parent2)]
    return child1, child2




Single-point Crossover: [64, 64, 128] [32, 128, 256, 512]
Two-point Crossover: [64, 64, 256, 512] [32, 128, 128]
Uniform Crossover: [32, 128, 128] [64, 64, 256]
Arithmetic Crossover: [33.088962211352, 66.177924422704, 132.355848845408] [62.911037788648, 125.822075577296, 251.644151154592]
Random Resetting Mutation: [80, 128, 256, 512]
Swap Mutation: [64, 128, 512, 256]
Shrink Mutation: [64, 128, 256]
Grow Mutation: [64, 128, 185, 256, 512]


In [ ]:
# Mutation functions
def random_resetting_mutation(individual, min_size, max_size):
    mutated = individual.copy()
    index = random.randint(0, len(mutated) - 1)
    mutated[index] = random.randint(min_size, max_size)
    return mutated

def swap_mutation(individual):
    if len(individual) < 2:
        return individual  # No mutation if there are less than 2 layers
    index1, index2 = random.sample(range(len(individual)), 2)
    mutated = individual.copy()
    mutated[index1], mutated[index2] = mutated[index2], mutated[index1]
    return mutated

def shrink_mutation(individual):
    if len(individual) <= 1:
        return individual  # No mutation if there's only one layer
    index = random.randint(0, len(individual) - 1)
    mutated = individual[:index] + individual[index+1:]
    return mutated

def grow_mutation(individual, min_size, max_size):
    index = random.randint(0, len(individual))
    new_layer_size = random.randint(min_size, max_size)
    mutated = individual[:index] + [new_layer_size] + individual[index:]
    return mutated


Here 
- A CX operators that change also the number of layers of the nn

In [2]:
import random

def variable_length_single_point_crossover(parent1, parent2):
    point1 = random.randint(1, len(parent1) - 1)
    point2 = random.randint(1, len(parent2) - 1)
    child1 = parent1[:point1] + parent2[point2:]
    child2 = parent2[:point2] + parent1[point1:]
    return child1, child2

def variable_length_two_point_crossover(parent1, parent2):
    if len(parent1) < 2 or len(parent2) < 2:
        return parent1, parent2  # No crossover if either parent has less than 2 layers
    point1_1 = random.randint(1, len(parent1) - 2)
    point1_2 = random.randint(point1_1 + 1, len(parent1) - 1)
    point2_1 = random.randint(1, len(parent2) - 2)
    point2_2 = random.randint(point2_1 + 1, len(parent2) - 1)
    child1 = parent1[:point1_1] + parent2[point2_1:point2_2] + parent1[point1_2:]
    child2 = parent2[:point2_1] + parent1[point1_1:point1_2] + parent2[point2_2:]
    return child1, child2

def hybrid_crossover(parent1, parent2, min_size, max_size):
    point1 = random.randint(1, len(parent1) - 1)
    point2 = random.randint(1, len(parent2) - 1)
    new_layer = random.randint(min_size, max_size)
    child1 = parent1[:point1] + [new_layer] + parent2[point2:]
    child2 = parent2[:point2] + [new_layer] + parent1[point1:]
    return child1, child2

# Example usage
parent1 = [64, 128, 256, 512]
parent2 = [32, 64, 128]

# Variable-length crossover
child1, child2 = variable_length_single_point_crossover(parent1, parent2)
print("Children after variable-length single-point crossover:", child1, child2)

child1, child2 = variable_length_two_point_crossover(parent1, parent2)
print("Children after variable-length two-point crossover:", child1, child2)

child1, child2 = hybrid_crossover(parent1, parent2, 16, 512)
print("Children after hybrid crossover:", child1, child2)


Children after variable-length single-point crossover: [64, 128, 128] [32, 64, 256, 512]
Children after variable-length two-point crossover: [64, 128, 64, 512] [32, 256, 128]
Children after hybrid crossover: [64, 193, 64, 128] [32, 193, 128, 256, 512]
